# R5 — Writeup Tables (Rust)

Mirrors `5_writeup_tables.ipynb`. Generates summary tables for the Rust analysis:
- Table 1: Concept fraction across the 3x3 parameter grid
- Table 2: Loudest neurons at mid-layer
- Table 3: Layer-by-layer concept fraction profile

In [ ]:
# Cell 1 – Configuration & load
import subprocess, sys, os
for pkg in ["numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
from pathlib import Path

EPSILONS = ["0.001", "0.1", "0.5"]

LANG = "R"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
CONSISTENCIES = ["0.2", "0.5", "0.8"]
N_LAYERS = 28
ALL_LAYERS_STR = "_".join(str(l) for l in range(N_LAYERS))

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = Path("/content/drive/MyDrive/DATA/New-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/New-Atlas")


def _fname(eps: str, cons: str) -> str:
    return f"{PREFIX}4_neuron_list_eps{eps}_cons{cons}_L{ALL_LAYERS_STR}_both.csv"


# Rust-specific object classification
# Construct keywords that are syntactically unique (high expected concept fraction)
MODULAR_CONSTRUCTS = {
    "rust__Use", "rust__Mod", "rust__Break", "rust__Continue",
    "rust__Return", "rust__Unsafe", "rust__Await",
}


def classify(obj: str) -> str:
    """Classify a Rust object into one of three groups."""
    # Objects from RUST_OBJECT_FAMILIES (types, traits, primitives)
    # vs constructs from RUST_CONSTRUCT_FAMILIES
    # All use "rust__" prefix. Distinguish by checking known construct names.
    KNOWN_CONSTRUCTS = {
        "rust__For", "rust__While", "rust__Loop",
        "rust__If", "rust__Match",
        "rust__Fn", "rust__Closure",
        "rust__Let", "rust__LetMut", "rust__Const", "rust__Static",
        "rust__Struct", "rust__Enum", "rust__TypeAlias",
        "rust__Impl", "rust__Trait",
        "rust__Use", "rust__Mod",
        "rust__Return", "rust__Break", "rust__Continue",
        "rust__Async", "rust__Await",
        "rust__Unsafe",
        "rust__Ref", "rust__RefMut", "rust__Deref",
        "rust__Lifetime",
        "rust__Macro", "rust__Attribute",
        "rust__QuestionMark",
    }
    if obj in MODULAR_CONSTRUCTS:
        return "Modular Construct"
    if obj in KNOWN_CONSTRUCTS:
        return "Non-modular Construct"
    return "Object"  # types, traits, primitives


# Load all available CSV files
frames = {}
for eps in EPSILONS:
    for cons in CONSISTENCIES:
        path = DATA_DIR / _fname(eps, cons)
        if not path.exists():
            print(f"WARNING: missing {path.name}")
            continue
        df = pd.read_csv(path)
        df["circuit_size"] = df["n_concept_only"] + df["n_both"]
        df["group"] = df["object"].map(classify)
        frames[(eps, cons)] = df
        print(f"  Loaded {path.name}: {len(df)} rows")

print(f"\nLoaded {len(frames)} / {len(EPSILONS)*len(CONSISTENCIES)} files")

In [ ]:
# Cell 2 – Table 1: Concept fraction grid
print("TABLE 1: Concept Fraction across parameter grid")
print("=" * 60)

rows = []
for eps in EPSILONS:
    for cons in CONSISTENCIES:
        df = frames.get((eps, cons))
        if df is None:
            rows.append({"eps": eps, "Consistency": cons,
                         "Construct CF": None, "Object CF": None})
            continue

        construct_df = df[df["group"].isin(["Modular Construct", "Non-modular Construct"])]
        object_df = df[df["group"] == "Object"]

        c_cf = (construct_df["n_concept_only"].sum() / construct_df["circuit_size"].sum()
                if construct_df["circuit_size"].sum() > 0 else 0)
        o_cf = (object_df["n_concept_only"].sum() / object_df["circuit_size"].sum()
                if object_df["circuit_size"].sum() > 0 else 0)

        rows.append({
            "eps": eps, "Consistency": cons,
            "Construct CF": f"{c_cf:.1%}",
            "Object CF": f"{o_cf:.1%}",
        })

table1 = pd.DataFrame(rows)
display(table1)

In [ ]:
# Cell 3 – Table 2: Loudest neurons (eps=0.5, cons=0.8, layer 14)
MID_LAYER = 14

print(f"TABLE 2: Neuron counts at layer {MID_LAYER} (eps=0.5, cons=0.8)")
print("=" * 60)

df_loud = frames.get(("0.5", "0.8"))
if df_loud is not None:
    l_mid = df_loud[df_loud["layer"] == MID_LAYER].copy()

    rows2 = []
    for group_name in ["Modular Construct", "Non-modular Construct", "Object"]:
        group_data = l_mid[l_mid["group"] == group_name]
        c = group_data["n_concept_only"].sum()
        s = group_data["n_both"].sum()
        sz = c + s
        cf = c / sz if sz > 0 else 0
        rows2.append({
            "Group": group_name,
            "Concept-only": c,
            "Shared": s,
            "Circuit size": sz,
            "Concept fraction": f"{cf:.1%}",
        })

    table2 = pd.DataFrame(rows2)
    display(table2)
else:
    print("Missing eps=0.5, cons=0.8 data")

In [ ]:
# Cell 4 – Table 3: Layer profile (eps=0.001, cons=0.8)
print("TABLE 3: Layer-by-layer concept fraction (eps=0.001, cons=0.8)")
print("=" * 60)

df_base = frames.get(("0.001", "0.8"))
if df_base is not None:
    available_layers = sorted(df_base["layer"].unique())

    rows3 = []
    for group_label in ["Modular Construct", "Non-modular Construct", "Object"]:
        row = {"Group": group_label}
        for layer in available_layers:
            sub = df_base[(df_base["layer"] == layer) & (df_base["group"] == group_label)]
            c = sub["n_concept_only"].sum()
            sz = sub["circuit_size"].sum()
            cf = c / sz if sz > 0 else 0
            row[f"L{layer}"] = f"{cf:.1%}"
        rows3.append(row)

    table3 = pd.DataFrame(rows3)
    display(table3)
else:
    print("Missing eps=0.001, cons=0.8 data")

In [ ]:
# Cell 5 – Per-object detail (eps=0.5, cons=0.8)
print("PER-OBJECT DETAIL (eps=0.5, cons=0.8)")
print("=" * 60)

if df_loud is not None:
    summary = (df_loud.groupby(["object", "group"])
               .agg(mean_concept_only=("n_concept_only", "mean"),
                    mean_both=("n_both", "mean"),
                    total_concept_only=("n_concept_only", "sum"),
                    total_circuit=("circuit_size", "sum"))
               .reset_index())
    summary["concept_fraction"] = (summary["total_concept_only"] / summary["total_circuit"]).fillna(0)
    summary = summary.sort_values("concept_fraction", ascending=False)
    display(summary.head(30))
else:
    print("Missing data")

print("\nR5 complete.")